# 01 — Validação e preparação do HMDA 2024

No Colab, coloque o CSV nacional extraído no Google Drive em **Meu Drive/hmda-credit-fairness/data/raw/2024_public_lar_csv.csv**. Se preferir, altere CSV_PATH para apontar para o arquivo disponível. Execute as células em ordem. O Parquet e as tabelas de auditoria ficam na mesma pasta de projeto, em data/processed.

A leitura usa blocos e seleciona as colunas necessárias. A fonte e os códigos estão no [dicionário público do LAR](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields).

In [17]:
from pathlib import Path
from collections import Counter

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    CSV_PATH = ROOT / 'data/raw/2024_public_lar_csv/2024_public_lar_csv.csv'
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hmda-credit-fairness')
    CSV_PATH = ROOT / 'data/raw/2024_public_lar_csv.csv'
OUTPUT = ROOT / 'data/processed'
CHUNK_SIZE = 100_000

if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Envie o CSV extraído ou ajuste CSV_PATH: {CSV_PATH}')

OUTPUT.mkdir(parents=True, exist_ok=True)

## Recorte e tipos

Incluímos compras de residência principal com decisão de aprovação ou recusa. O alvo vale 1 para os códigos 1 e 2 e 0 para o código 3. Idade e razão dívida/renda ficam como categorias, pois o LAR público contém faixas. Raça e etnia usam as categorias agregadas publicadas pelo HMDA, que podem refletir informações do solicitante e do co-solicitante. Para a renda relativa do setor, valores não positivos são tratados como ausentes nesta versão do protocolo.

In [18]:
BASE_NUM = ['income', 'loan_amount', 'loan_term', 'property_value']
BASE_CAT = ['lei', 'activity_year', 'applicant_age', 'debt_to_income_ratio',
            'loan_type', 'lien_status', 'construction_method', 'total_units']
DEMOGRAPHIC = ['derived_race', 'derived_ethnicity']
TRACT_INCOME = 'tract_to_msa_income_percentage'
ELIGIBILITY = ['loan_purpose', 'occupancy_type', 'action_taken']
FEATURES = BASE_NUM + BASE_CAT + DEMOGRAPHIC + [TRACT_INCOME]
INPUT_COLUMNS = FEATURES + ELIGIBILITY

header = pd.read_csv(CSV_PATH, nrows=0).columns
missing_columns = set(INPUT_COLUMNS) - set(header)
if missing_columns:
    raise ValueError(f'Colunas obrigatórias ausentes: {sorted(missing_columns)}')

schema = pa.schema(
    [(name, pa.float64()) for name in BASE_NUM]
    + [(name, pa.string()) for name in BASE_CAT + DEMOGRAPHIC]
    + [(TRACT_INCOME, pa.float64()), ('approved', pa.int8())]
)

In [19]:
counts = Counter({key: 0 for key in ['lidos', 'hmda_2024', 'compra',
                                     'residencia_principal', 'decisao_elegivel']})
actions = Counter()
target_counts = Counter()
missing_counts = Counter({name: 0 for name in FEATURES})
categories = {name: Counter() for name in DEMOGRAPHIC}
parquet_path = OUTPUT / 'hmda_2024_purchase.parquet'
partial = parquet_path.with_suffix('.parquet.part')

with pq.ParquetWriter(partial, schema, compression='zstd') as writer:
    chunks = pd.read_csv(
        CSV_PATH, usecols=INPUT_COLUMNS, dtype='string', chunksize=CHUNK_SIZE,
        na_values=['Exempt', 'exempt', 'Not Applicable'],
    )
    for chunk in chunks:
        counts['lidos'] += len(chunk)
        year = chunk.loc[chunk['activity_year'].eq('2024')]
        counts['hmda_2024'] += len(year)
        purchase = year.loc[year['loan_purpose'].eq('1')]
        counts['compra'] += len(purchase)
        primary = purchase.loc[purchase['occupancy_type'].eq('1')]
        counts['residencia_principal'] += len(primary)
        actions.update(primary['action_taken'].fillna('Ausente').value_counts().to_dict())

        eligible = primary['action_taken'].isin(['1', '2', '3'])
        selected = primary.loc[eligible, FEATURES].copy()
        counts['decisao_elegivel'] += len(selected)
        if selected.empty:
            continue

        for name in BASE_NUM + [TRACT_INCOME]:
            selected[name] = pd.to_numeric(selected[name], errors='coerce')
            selected[name] = selected[name].where(~selected[name].isin([float('inf'), -float('inf')]))
        selected[TRACT_INCOME] = selected[TRACT_INCOME].where(selected[TRACT_INCOME].gt(0))
        selected['approved'] = primary.loc[eligible, 'action_taken'].isin(['1', '2']).astype('int8')
        target_counts.update(selected['approved'].value_counts().to_dict())
        missing_counts.update(selected[FEATURES].isna().sum().to_dict())
        for name in DEMOGRAPHIC:
            categories[name].update(selected[name].fillna('Ausente').value_counts().to_dict())

        writer.write_table(pa.Table.from_pandas(selected, schema=schema, preserve_index=False))

if set(target_counts) != {0, 1}:
    raise ValueError('O recorte precisa conter solicitações aprovadas e recusadas.')

assert pq.ParquetFile(partial).metadata.num_rows == counts['decisao_elegivel'] == sum(target_counts.values())
partial.replace(parquet_path)
print(f'Parquet pronto: {parquet_path}')

Parquet pronto: c:\Users\cg3034593\Documents\artigo\data\processed\hmda_2024_purchase.parquet


In [20]:
def save_counts(values, label, filename, denominator=None):
    result = pd.Series(dict(values), dtype='int64', name='n').rename_axis(label).reset_index()
    if denominator:
        result['proporcao'] = result['n'] / denominator
    result.to_csv(OUTPUT / filename, index=False)
    display(result)

save_counts(counts, 'etapa', 'validacao_etapas.csv')
save_counts(actions, 'action_taken', 'validacao_acoes.csv')
save_counts(target_counts, 'approved', 'validacao_alvo.csv', counts['decisao_elegivel'])
save_counts(missing_counts, 'campo', 'validacao_ausencias.csv', counts['decisao_elegivel'])
for name, values in categories.items():
    save_counts(values, name, f'validacao_{name}.csv', counts['decisao_elegivel'])

,etapa,n
0,lidos,12229298
1,hmda_2024,12229298
2,compra,6553105
3,residencia_principal,5864623
4,decisao_elegivel,3794116


,action_taken,n
0,1,3096020
1,6,1004583
2,3,557514
3,4,705753
4,2,140582
5,5,171112
6,8,144092
7,7,44967


,approved,n,proporcao
0,1,3236602,0.853058
1,0,557514,0.146942


,campo,n,proporcao
0,income,28908,0.007619
1,loan_amount,0,0.000000
2,loan_term,88533,0.023334
3,property_value,140683,0.037079
4,lei,0,0.000000
5,activity_year,0,0.000000
6,applicant_age,0,0.000000
7,debt_to_income_ratio,118947,0.031350
8,loan_type,0,0.000000
9,lien_status,0,0.000000


,derived_race,n,proporcao
0,White,2477427,0.652966
1,Race Not Available,544323,0.143465
2,Black or African American,348564,0.091870
3,Asian,268671,0.070813
4,Joint,105771,0.027878
5,American Indian or Alaska Native,30394,0.008011
6,2 or more minority races,10609,0.002796
7,Native Hawaiian or Other Pacific Islander,7706,0.002031
8,Free Form Text Only,651,0.000172


,derived_ethnicity,n,proporcao
0,Not Hispanic or Latino,2615551,0.689370
1,Ethnicity Not Available,484785,0.127773
2,Hispanic or Latino,571394,0.150600
3,Joint,121127,0.031925
4,Free Form Text Only,1259,0.000332
